# Cross-setup LPB evaluation

Visualize LPB coverage and bound size when the survival model is trained on one setup and calibration/test data come from another setup over the same base dataset.

In [ ]:
from pathlib import Path
import sys
import textwrap

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.safety_evaluation.cross_setup_utils import get_cross_setup_experiment_name


config = {
    "dataset_name": "dataset_red_team",
    "model_dataset_setup": "attack_default_attack_qwen25_14b_instruct_lm_target_qwen25_14b_instruct_judge_llm-judge_qwen25_14b_instruct",
    "evaluation_dataset_setup": "attack_default_attack_qwen25_14b_instruct_lm_target_gemma3_4b_it_judge_llm-judge_qwen25_14b_instruct",
    "budget_per_sample": 20.0,
    "cal_size": 3000,
    "tau_prior": 0.56,
    "gamma": 10.0,
}

experiment_name = get_cross_setup_experiment_name(
    config["dataset_name"],
    config["model_dataset_setup"],
    config["evaluation_dataset_setup"],
    config["budget_per_sample"],
    config["cal_size"],
    config["tau_prior"],
    config["gamma"],
)
csv_path = (
    PROJECT_ROOT
    / "results"
    / "merged_calibration_dfs"
    / experiment_name
    / "all_df.csv"
)
csv_path


In [ ]:
if not csv_path.exists():
    raise FileNotFoundError(
        f"Merged cross-setup LPB results were not found at {csv_path}. "
        "Run src/safety_evaluation/scripts/cross_setup_lpb.sh first."
    )

all_df = pd.read_csv(csv_path)
required_columns = {
    "seed",
    "calibration_name",
    "target_coverage",
    "coverage",
    "size",
    "dataset_name",
    "model_dataset_setup",
    "evaluation_dataset_setup",
}
missing_columns = required_columns.difference(all_df.columns)
if missing_columns:
    raise KeyError(f"Missing required columns: {sorted(missing_columns)}")

expected_metadata = {
    "dataset_name": config["dataset_name"],
    "model_dataset_setup": config["model_dataset_setup"],
    "evaluation_dataset_setup": config["evaluation_dataset_setup"],
}
for column, expected_value in expected_metadata.items():
    observed_values = set(all_df[column].dropna().unique())
    if observed_values != {expected_value}:
        raise ValueError(
            f"Unexpected {column}: expected {expected_value!r}, observed {sorted(observed_values)!r}"
        )

display_names = {
    "uncalibrated": "Uncalibrated",
    "oracle_survival_calibration": "Oracle calibration",
    "calibration_optimized_allocation": "Static allocation",
    "calibration_adaptive_optimized_allocation": "Locally adaptive",
    "calibration_projected_optimization_platt_prob_allocation": "DAPRO",
}
plot_df = all_df.loc[all_df["calibration_name"].isin(display_names)].copy()
plot_df["Method"] = plot_df["calibration_name"].map(display_names)
plot_df["Target coverage (%)"] = 100 * plot_df["target_coverage"]
plot_df["Coverage (%)"] = 100 * plot_df["coverage"]

method_order = [name for name in display_names.values() if name in set(plot_df["Method"])]
if not method_order:
    raise ValueError(
        f"None of the requested methods are present. Available: {sorted(all_df['calibration_name'].unique())}"
    )

print(f"Loaded {plot_df['seed'].nunique()} seeds and {len(method_order)} methods.")
plot_df.groupby("Method")["seed"].nunique().reindex(method_order).rename("number_of_seeds")


In [ ]:
sns.set_theme(style="whitegrid", context="talk")
palette = dict(zip(method_order, sns.color_palette("colorblind", len(method_order))))

fig, axes = plt.subplots(1, 2, figsize=(17, 6.5), sharex=True)

for method in method_order:
    method_df = plot_df.loc[plot_df["Method"].eq(method)]
    stats = method_df.groupby("Target coverage (%)", as_index=False).agg(
        coverage_mean=("Coverage (%)", "mean"),
        coverage_std=("Coverage (%)", "std"),
        size_mean=("size", "mean"),
        size_std=("size", "std"),
    ).fillna(0)
    x = stats["Target coverage (%)"].to_numpy()
    coverage_mean = stats["coverage_mean"].to_numpy()
    coverage_std = stats["coverage_std"].to_numpy()
    size_mean = stats["size_mean"].to_numpy()
    size_std = stats["size_std"].to_numpy()
    color = palette[method]

    axes[0].plot(x, coverage_mean, label=method, color=color, linewidth=2.5)
    axes[0].fill_between(
        x,
        coverage_mean - coverage_std,
        coverage_mean + coverage_std,
        color=color,
        alpha=0.15,
    )
    axes[1].plot(x, size_mean, label=method, color=color, linewidth=2.5)
    axes[1].fill_between(
        x,
        size_mean - size_std,
        size_mean + size_std,
        color=color,
        alpha=0.15,
    )

coverage_range = np.sort(plot_df["Target coverage (%)"].unique())
axes[0].plot(
    coverage_range,
    coverage_range,
    color="dimgray",
    linestyle="--",
    linewidth=2,
    label="Target",
)

axes[0].set_ylabel("Empirical coverage (%)")
axes[0].set_title("Coverage under setup shift", loc="left")
axes[1].set_ylabel("Mean LPB size")
axes[1].set_title("Bound size under setup shift", loc="left")
for ax in axes:
    ax.set_xlabel("Target coverage (%)")
    ax.grid(True, alpha=0.3)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    labels,
    loc="center left",
    bbox_to_anchor=(1.0, 0.5),
    title="Method",
    frameon=True,
)
def target_model_label(setup):
    marker = "_lm_target_"
    if marker in setup and "_judge_" in setup:
        return setup.split(marker, 1)[1].split("_judge_", 1)[0]
    return textwrap.shorten(setup, width=42, placeholder="…")

source_label = target_model_label(config["model_dataset_setup"])
evaluation_label = target_model_label(config["evaluation_dataset_setup"])
fig.suptitle(
    "Cross-setup LPB: source-trained model evaluated on a different setup\n"
    f"model setup target: {source_label}  →  calibration/test setup target: {evaluation_label}",
    y=1.04,
)

figure_dir = PROJECT_ROOT / "notebooks" / "figures" / "cross_setup" / experiment_name
figure_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_dir / "cross_setup_lpb_coverage_and_size.png"
fig.tight_layout()
fig.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved figure to {figure_path}")
